In [ ]:
pip install lxml


In [21]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = "You are an expert financial analyst specializing in cryptocurrency and macroeconomic trends. Your job is to fetch the latest macroeconomic news impacting the crypto market, summarize key points, and provide a well-reasoned opinion on potential market implications."

In [ ]:
# imports

import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [ ]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

openai = OpenAI()

In [ ]:
#testing openAI

message = "Hello,GPT!"
response = openai.chat.completions.create(
    model = "gpt-4o-mini" , 
    messages = [{"role":"user", "content":message}])
print(response.choices[0].message.content)

In [ ]:
class Website:
    def __init__(self, url):
        self.url = url
        self.title = None
        self.text = None
        self._fetch_content()

    def _fetch_content(self):
        """Fetches and processes the webpage content."""
        headers = {"User-Agent": "Mozilla/5.0"}
        
        try:
            response = requests.get(self.url, headers=headers, timeout=10)
            response.raise_for_status()  # Raise error for bad status codes

            soup = BeautifulSoup(response.content, "lxml")  # Use lxml for better performance
            self.title = soup.title.string.strip() if soup.title else "No title found"

            self._clean_soup(soup)
            self.text = soup.get_text(separator="\n", strip=True)

        except requests.exceptions.RequestException as e:
            print(f"Request error: {e}")
            self.text = "Failed to fetch content"
        except Exception as e:
            print(f"Parsing error: {e}")
            self.text = "Failed to parse content"

    def _clean_soup(self, soup):
        """Removes unnecessary elements from the parsed content."""
        if soup.body:
            for tag in soup.body(["script", "style", "img", "input", "iframe", "noscript", "form", "svg"]):
                tag.decompose()  # Removes tag and its content

    def __str__(self):
        """Returns a string representation of the website object."""
        return f"Website Title: {self.title}\nURL: {self.url}\nExtracted Text: {self.text[:500]}..."  # Truncate text for display


In [ ]:
def analyze_news(title, source, content):
    system_prompt = """
    You are an AI-powered Fake News Detector. Your job is to analyze news articles and determine if they are likely to be fake or real. 
    Consider:
    - The credibility of the source (official news sites vs. suspicious blogs).
    - Writing style (sensationalism, exaggerated claims, lack of references).
    - Cross-checking claims with fact-checked databases.
    - Bias detection in language used.

    Your response must include:
    - A classification: [Fake] or [Real]
    - A reasoning summary (why you classified it as fake or real)
    - A confidence score (0-100%) in your decision.
    """

    user_prompt = f"""
    Analyze the following news article and determine if it is likely to be fake or real. Consider credibility, writing style, and factual accuracy.

    Title: {title}
    Source: {source}
    Content: {content}

    Provide a classification, a reasoning summary, and a confidence score.
    """

    response = openai.chat.completions.create(
        model="gpt-4o-mini",  # Adjust model based on API provider
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content

# # Example usage
# title = "Breaking: Aliens Have Landed in New York!"
# source = "UFOConspiracyNews.com"
# content = "According to an anonymous source, extraterrestrial beings were spotted in Central Park."

# result = analyze_news(title, source, content)
# print(result)


### Testing real news

In [ ]:
# Let's try one out. Change the website and add print statements to follow along.
web = 'https://edition.cnn.com/2025/02/22/business/luxury-brands-middle-income-aspirational-consumer/index.html'
ed = Website(web)
# print(ed.title)
# print('\n')
# print(ed.text)

result = analyze_news(ed.title, web, ed.text)
print(result)

### Testing Fake news

In [ ]:
title = "Share to Get Money from Bill Gates"
web = "https://library-nd.libguides.com/fakenews/examples"
news = """

Share a certain post of Bill Gates on Facebook and he will send you money.
"Hey Facebook, As some of you may know, I'm Bill Gates. If you click that share link, I will give you $5,000. I always deliver, I mean, I brought you Windows XP, right?"

"""
result = analyze_news(title, web, ed.text)
print(result)